In [ ]:
!git clone https://github.com/NhuGiap04/Fk-Diffusion-Steering.git

In [ ]:
%cd Fk-Diffusion-Steering
!git pull

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
%cd text_to_image

In [ ]:
import torch
from diffusers import DDIMScheduler

from XL_diffusers import TemperedDiverseRejuvenatedStableDiffusionXL

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = TemperedDiverseRejuvenatedStableDiffusionXL.from_pretrained(
  "stabilityai/stable-diffusion-xl-base-1.0",
  torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

In [ ]:
import math
import torch
import matplotlib.pyplot as plt
from XL_diffusers import latent_to_decode


def display_tdr_timestep_inference(
    pipe,
    call_kwargs,
    image_index=0,
    frame_stride=5,
    max_frames_to_show=24,
    nrows=2,
    decode_dtype=None,
):
    """Run TDR inference, capture per-step latents, and visualize decoded timesteps.

    Args:
        pipe: TemperedDiverseRejuvenatedStableDiffusionXL pipeline instance.
        call_kwargs: Dict of kwargs passed to pipe(...).
        image_index: Which image index from each decoded step to display.
        frame_stride: Keep every Nth step (and always the last).
        max_frames_to_show: Cap number of displayed frames.
        nrows: Number of rows in display grid.
        decode_dtype: Optional dtype used for VAE decode (defaults to pipe.vae.dtype).
    Returns:
        result: pipeline output object.
        frame_records: list of dicts with step metadata and decoded images.
    """
    if frame_stride < 1:
        raise ValueError("frame_stride must be >= 1")
    if max_frames_to_show < 1:
        raise ValueError("max_frames_to_show must be >= 1")

    step_latents = []
    step_timesteps = []
    step_indices = []

    def capture_callback(_pipe, step_idx, timestep, callback_kwargs):
        step_latents.append(callback_kwargs["latents"].detach().cpu())
        step_timesteps.append(int(timestep))
        step_indices.append(int(step_idx))
        return callback_kwargs

    run_kwargs = dict(call_kwargs)
    result = pipe(
        **run_kwargs,
        callback_on_step_end=capture_callback,
        callback_on_step_end_tensor_inputs=["latents"],
    )

    if len(step_latents) == 0:
        print("No timestep latents were captured.")
        return result, []

    latent_dtype = decode_dtype or pipe.vae.dtype
    frame_records = []

    with torch.no_grad():
        for i, latents_cpu in enumerate(step_latents):
            latents = latents_cpu.to(device=pipe.device, dtype=latent_dtype)
            decoded = latent_to_decode(model=pipe, output_type="pil", latents=latents)
            pil_images = pipe.image_processor.postprocess(decoded, output_type="pil")
            frame_records.append(
                {
                    "step_idx": step_indices[i],
                    "timestep": step_timesteps[i],
                    "images": pil_images,
                }
            )

    n_steps = len(frame_records)
    if image_index < 0 or image_index >= len(frame_records[0]["images"]):
        raise IndexError(
            f"image_index={image_index} out of range for batch size {len(frame_records[0]['images'])}"
        )

    keep = [i for i in range(n_steps) if (i % frame_stride == 0) or (i == n_steps - 1)]
    if len(keep) > max_frames_to_show:
        keep_idx = torch.linspace(0, len(keep) - 1, steps=max_frames_to_show).round().long().tolist()
        keep = [keep[k] for k in keep_idx]

    n_display = len(keep)
    ncols = max(1, math.ceil(n_display / nrows))
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4 * ncols, 4 * nrows),
        squeeze=False,
    )
    axes = axes.flatten()

    for ax_i, ax in enumerate(axes):
        if ax_i < n_display:
            rec = frame_records[keep[ax_i]]
            ax.imshow(rec["images"][image_index])
            ax.set_title(
                f"step={rec['step_idx']} | t={rec['timestep']}",
                fontsize=9,
            )
        ax.axis("off")

    fig.suptitle("TemperedDiverseRejuvenatedSDXL timestep inference", fontsize=13)
    plt.tight_layout()
    plt.show()

    return result, frame_records

In [ ]:
prompts = [
    "A photo of a brown knife and a blue donut",
] * 12

# Example usage
result, frames = display_tdr_timestep_inference(
    pipe,
    call_kwargs={
        "prompt": prompts,
        "num_inference_steps": 40,
        "guidance_scale": 5.0,
        "num_images_per_prompt": 1,
        "tdr_num_islands": 3,
        "tdr_target_lambda": 2.0,
        "tdr_particles_per_island": [4, 4, 4],
        "tdr_resample_frequency": 5,
        "tdr_ess_threshold_ratio": 0.5,
        "tdr_rollout_count": 2,
        "tdr_rollout_steps": 2,
        "tdr_rejuvenation_steps": 1,
        "tdr_rejuvenation_noise_scale": 0.08,
        "tdr_immigrant_fraction": 0.15,
        "tdr_final_select_m": 4,
        "tdr_final_diversity_beta": 0.2,
        "tdr_reward_fn": "ImageReward",
        "tdr_metric_to_chase": "overall_score",
    },
    image_index=0,
    frame_stride=4,
    max_frames_to_show=20,
    nrows=2,
)

print(f"Captured {len(frames)} denoising steps.")